# Grid-based Boomerang — quick sampling demo

Sample from `make_gaussian`, `make_banana`, and `make_gaussian_mixture` (from `sazz.models.math_targets`, imported read-only) using the new `GridBoomerangSampler` (`sazz/gpu_friendly/samplers/grid_boomerang.py`), which uses the Andral & Kamatani (2024) grid-based piecewise-constant upper bound instead of Brent/PLI.

This notebook only exercises the new `sazz/gpu_friendly/` tree — nothing in `sazz/samplers/` or `sazz/models/` is modified. The analytic `math_targets.py` closures are pure functions of `beta` (no detaching), so they compose directly with the grid sampler's `torch.func.grad`/`vmap`/`jvp`-based bound construction without any adapter.

In [ ]:
import os
from pathlib import Path

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.models.math_targets import make_gaussian, make_banana, make_gaussian_mixture
from sazz.gpu_friendly.samplers.grid_boomerang import GridBoomerangSampler
from sazz.gpu_friendly.samplers.grid_zigzag import GridZigZagSampler
from sazz.gpu_friendly.utils.resample import resample_zigzag_path_torch, resample_boomerang_path_torch

In [ ]:
def run_grid_boomerang(target, N=50_000, refresh_rate=1.0, n_segments=20,
                        grid_t_max_init=math.pi / 4, dtype=torch.float64):
    """Build a GridBoomerangSampler for `target` and run it for N skeleton points."""
    sampler = GridBoomerangSampler(
        grad_target=target.grad_target,
        D=target.D,
        refresh_rate=refresh_rate,
        grid_t_max_init=grid_t_max_init,
        n_segments=n_segments,
        dtype=dtype,
    )
    sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)
    result = sampler.sample(N=N, diagnostics=True)
    result["sampler"] = sampler
    return result

def run_grid_zigzag(target, N=50_000, n_segments=20,
                        grid_t_max_init=1.0, dtype=torch.float64):
    """Build a GridBoomerangSampler for `target` and run it for N skeleton points."""
    sampler = GridZigZagSampler(
        grad_target=target.grad_target,
        D=target.D,
        gamma=0.01,
        grid_t_max_init=grid_t_max_init,
        n_segments=n_segments,
        dtype=dtype,
    )
    result = sampler.sample(N=N, x0=target.x_ref, diagnostics=True)
    result["sampler"] = sampler
    return result

def plot_marginals(target, result, model="zigzag", max_coords=4, bins=60, burnin_frac=0.5):
    """Histogram of each coordinate vs the analytic marginal PDF, plus the
    adaptive grid_t_max trajectory over the run."""
    coords = list(target.marginal_grids.keys())[:max_coords]
    # Resamplers require torch.Tensor inputs (they read .dtype/.device off
    # them directly) -- keep these as tensors here, only convert to numpy
    # for matplotlib after resampling.
    positions = result["positions"]
    velocities = result["velocities"]
    times = result["times"]
    n = positions.shape[0]
    if model == "zigzag":
        samples = resample_zigzag_path_torch(positions, velocities, times, N_resample=10_000, burnin_frac=burnin_frac)
    elif model == "boomerang":
        samples = resample_boomerang_path_torch(positions, velocities, times, target.x_ref, N_resample=10_000, burnin_frac=burnin_frac)
    else:
        raise ValueError(f"Please provide a valid model ('zigzag' or 'boomerang'), got {model!r}")
    samples = samples.numpy()

    fig, axes = plt.subplots(1, len(coords) + 1, figsize=(3.2 * (len(coords) + 1), 3))

    for i, c in enumerate(coords):
        ax = axes[i]
        info = target.marginal_grids[c]
        ax.hist(samples[:, c], bins=bins, density=True, alpha=0.5, label="grid boomerang")
        ax.plot(info["grid"], info["pdf"], "k--", lw=1.5, label="analytic")
        ax.set_title(info["label"])
        if i == 0:
            ax.legend(fontsize=8)

    ax = axes[-1]
    ax.plot(result["grid_t_max_log"])
    ax.set_title("adaptive grid_t_max")
    ax.set_xlabel("iteration")

    fig.tight_layout()
    plt.show()

    print(f"bound_violations: {result['bound_violations']}")
    print(f"gradient_evals: {result['gradient_evals']} "
          f"({result['gradient_evals'] / n:.1f} / skeleton point)")

In [ ]:
def plot_marginals_joint(target, result_list, max_coords=4, bins=60, burnin_frac=0.5):
    """Histogram of each coordinate vs the analytic marginal PDF, with
    zigzag and boomerang samples overlaid on the same axes."""
    coords = list(target.marginal_grids.keys())[:max_coords]

    fig, axes = plt.subplots(1, len(coords), figsize=(3.2 * len(coords), 3))
    if len(coords) == 1:
        axes = [axes]

    for result in result_list:
        # Resamplers require torch.Tensor inputs (they read .dtype/.device
        # off them directly) -- keep these as tensors here, only convert to
        # numpy for matplotlib after resampling.
        positions = result["positions"]
        velocities = result["velocities"]
        times = result["times"]

        if isinstance(result["sampler"], GridZigZagSampler):
            samples = resample_zigzag_path_torch(positions, velocities, times, N_resample=10_000, burnin_frac=burnin_frac)
            label = "Zigzag"
        elif isinstance(result["sampler"], GridBoomerangSampler):
            samples = resample_boomerang_path_torch(positions, velocities, times, target.x_ref, N_resample=10_000, burnin_frac=burnin_frac)
            label = "Boomerang"
        else:
            raise ValueError(f"Unrecognized sampler type: {type(result['sampler'])!r}")
        samples = samples.numpy()

        for i, c in enumerate(coords):
            axes[i].hist(samples[:, c], bins=bins, density=True, alpha=0.5, label=label)

        n = positions.shape[0]
        print(f"[{label}] bound_violations: {result['bound_violations']}")
        print(f"[{label}] gradient_evals: {result['gradient_evals']} "
              f"({result['gradient_evals'] / n:.1f} / skeleton point)")

    for i, c in enumerate(coords):
        info = target.marginal_grids[c]
        axes[i].plot(info["grid"], info["pdf"], "k--", lw=1.5, label="Analytic")
        axes[i].set_title(info["label"])
        if i == 0:
            axes[i].legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(fname=f"results/plots/math_targets/{target.name}.png")
    plt.show()


## 0. Gaussian (diagonal)

The reference measure equals the target, so the excess gradient is identically zero. The grid Boomerang should report **zero bounces** (and, correspondingly, `bound_violations == 0` since the rate is exactly zero everywhere).

In [ ]:
target_gauss = make_gaussian(D=5, cov="diagonal")
print(f"Target: {target_gauss.name}  D={target_gauss.D}")
result_gauss_zigzag = run_grid_zigzag(target_gauss, N=50_000)
result_gauss_boom = run_grid_boomerang(target_gauss, N=50_000)

In [ ]:
# plot_marginals(target_gauss, result_gauss_zigzag, model="zigzag", max_coords=3)
# plot_marginals(target_gauss, result_gauss_boom, model="boomerang", max_coords=3)

In [ ]:
result_list = [result_gauss_zigzag, result_gauss_boom]
plot_marginals_joint(target_gauss, result_list, max_coords=3)


## 1. Gaussian (non-diagonal)

A dense `Sigma_inv` still matching the target exactly — same zero-bounce expectation, but exercises the full-matrix path in `preprocess`/`grad_U_excess`/`reflect_velocity` instead of the diagonal shortcut.

In [ ]:
target_gauss_dense = make_gaussian(D=5, cov="random")
print(f"Target: {target_gauss_dense.name}  D={target_gauss_dense.D}")
print(f"Sigma_inv is dense: {target_gauss_dense.Sigma_inv.ndim == 2}")
result_gauss_dense_zigzag = run_grid_zigzag(target_gauss_dense, N=50_000)
result_gauss_dense_boom = run_grid_boomerang(target_gauss_dense, N=50_000)

In [ ]:
# plot_marginals(target_gauss_dense, result_gauss_dense_zigzag, model="zigzag", max_coords=3)
# plot_marginals(target_gauss_dense, result_gauss_dense_boom, model="boomerang", max_coords=3)

In [ ]:
result_list_dense = [result_gauss_dense_zigzag, result_gauss_dense_boom]
plot_marginals_joint(target_gauss_dense, result_list_dense, max_coords=3)


## 2. Banana (Rosenbrock)

Non-Gaussian, curved geometry — the reference measure is only a rough match, so the excess gradient (and hence the rate) is genuinely non-trivial. Good stress test for the grid bound's tangent construction and the bound-violation safety net.

In [ ]:
target_banana = make_banana(a=1.0, scale=2.0)
print(f"Target: {target_banana.name}  D={target_banana.D}")
result_banana_zigzag = run_grid_zigzag(target_banana, N=50_000)
result_banana_boom = run_grid_boomerang(target_banana, N=50_000)

In [ ]:
# plot_marginals(target_banana, result_banana_zigzag, model="zigzag")
# plot_marginals(target_banana, result_banana_boom, model="boomerang")
result_list_banana = [result_banana_zigzag, result_banana_boom]
plot_marginals_joint(target_banana, result_list_banana, max_coords=3)


In [ ]:
a, scale = 1.0, 2.0

positions_zigzag = result_banana_zigzag["positions"].numpy()
positions_boom = result_banana_boom["positions"].numpy()

pad = 1.5
all_b0 = np.concatenate([positions_zigzag[:, 0], positions_boom[:, 0]])
all_b1 = np.concatenate([positions_zigzag[:, 1], positions_boom[:, 1]])
b0_grid = np.linspace(all_b0.min() - pad, all_b0.max() + pad, 300)
b1_grid = np.linspace(all_b1.min() - pad, all_b1.max() + pad, 300)
B0, B1 = np.meshgrid(b0_grid, b1_grid)
U = B0 / scale
E = 0.5 * U**2 + 0.5 * (B1 - a * U**2) ** 2
density = np.exp(-E)

# Log-spaced DENSITY levels: naturally denser near the peak (where density
# varies fast) and still reaches out into the tails, unlike either linear
# density levels (collapse near zero) or linear -E levels (peak
# under-resolved). Adjust the low end (1e-4 here) to pull the outermost
# ring further out/in.
levels = np.logspace(np.log10(density.max()) - 4, np.log10(density.max()), 12)

fig, ax = plt.subplots(figsize=(4, 4))
ax.contour(B0, B1, density, levels=levels, colors="black", linewidths=0.6, alpha=0.6)
ax.scatter(positions_zigzag[:, 0], positions_zigzag[:, 1], s=2, alpha=0.3, label="Zigzag")
ax.scatter(positions_boom[:, 0], positions_boom[:, 1], s=2, alpha=0.3, label="Boomerang")
#ax.set_title("Banana — skeleton")
ax.legend(fontsize=8)
fig.savefig(fname="results/plots/math_targets/banana_contour.png")
plt.show()


## 3. Gaussian mixture (bimodal)

Two well-separated modes — tests whether the grid sampler can cross the low-density valley, and whether the adaptive `grid_t_max` behaves sensibly when the rate's shape varies a lot across the space.

In [ ]:
target_mix = make_gaussian_mixture(D=1, preset="bimodal")
print(f"Target: {target_mix.name}  D={target_mix.D}")
result_mix_zigzag = run_grid_zigzag(target_mix, N=50_000)
result_mix_boom = run_grid_boomerang(target_mix, N=50_000, refresh_rate=2.0)

In [ ]:
# plot_marginals(target_mix, result_mix_zigzag, model="zigzag", max_coords=1)
# plot_marginals(target_mix, result_mix_boom, model="boomerang", max_coords=1)

In [ ]:
result_list_mix = [result_mix_zigzag, result_mix_boom]
plot_marginals_joint(target_mix, result_list_mix, max_coords=3)
